# GreenOps - results analysis

Explores the CSVs written by `python -m experiments.evaluation`. Run from the repository root (or the `notebooks/` folder).

In [ ]:
import os, sys
import pandas as pd
from IPython.display import Image, display

ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
EVAL = os.path.join(ROOT, 'results', 'evaluation')
FIGS = os.path.join(ROOT, 'results', 'figures')
pd.set_option('display.width', 200)

## E1 - main comparison (20 seeds x 2 seasons)

In [ ]:
e1 = pd.read_csv(os.path.join(EVAL, 'e1_main_summary.csv'))
cols = ['period', 'label', 'total_carbon_g_mean', 'total_carbon_g_std', 'total_cost_mean',
        'sla_violations_mean', 'average_latency_mean', 'p95_latency_mean',
        'carbon_vs_round_robin_pct_mean', 'carbon_vs_greenest_pct_mean']
e1[e1.period == 'both'][cols].round(2)

In [ ]:
pd.read_csv(os.path.join(EVAL, 'e1_significance.csv')).round(4)

In [ ]:
display(Image(os.path.join(FIGS, 'fig2_main_comparison.png')))

## E2 - Pareto frontier and hypervolume

In [ ]:
display(pd.read_csv(os.path.join(EVAL, 'e2_hypervolume.csv')).round(3))
display(Image(os.path.join(FIGS, 'fig3_pareto_frontier.png')))

## E3 - forecast-error sensitivity

In [ ]:
e3 = pd.read_csv(os.path.join(EVAL, 'e3_forecast_summary.csv'))
e3.pivot_table(index=['scenario', 'forecast_error'], columns='label', values='total_carbon_g_mean').round(0)

In [ ]:
display(Image(os.path.join(FIGS, 'fig4_forecast_sensitivity.png')))

## E4 - how much deferral is useful?

In [ ]:
e4 = pd.read_csv(os.path.join(EVAL, 'e4_maxwait_summary.csv'))
e4.pivot_table(index=['scenario', 'max_wait'], columns='label', values=['total_carbon_g_mean', 'p95_latency_mean']).round(1)

## E5 - load, E6 - extended regions, E7 - scalability

In [ ]:
e5 = pd.read_csv(os.path.join(EVAL, 'e5_load_summary.csv'))
e5.pivot_table(index='load_scale', columns='label', values='total_carbon_g_mean').round(0)

In [ ]:
pd.read_csv(os.path.join(EVAL, 'e6_extended_summary.csv')).filter(regex='label|_mean$').round(1)

In [ ]:
pd.read_csv(os.path.join(EVAL, 'e7_scale_runs.csv'))[['label', 'num_jobs', 'wall_seconds', 'ms_per_decision']].round(3)

## Re-run a single scenario interactively

In [ ]:
from experiments.scenario import load_traces, make_workload, make_scheduler, run_simulation
from experiments.evaluation import MOO_PARAMS

traces = load_traces('summer_2024')
summary, jobs = run_simulation(make_scheduler('greenops_moo', **MOO_PARAMS), traces,
                               make_workload(300, seed=0), forecast_error=0.15)
{k: round(v, 2) if isinstance(v, float) else v for k, v in summary.items()}